In [9]:
%load_ext autoreload
%autoreload 2

In [1]:
import sys
import os
import torch

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

In [2]:
from src.trainer import EWCSmoothTrainer
from src.data_utils import get_mnist_tasks, _extract_targets, get_context_sets
from src.utils.general import InContextHead, set_seed
from src import models
from src.regulariser import UnbiasRegulariser, L2Regulariser, MultiRegulariser

### Task Incremental Learning

In [6]:
SEED = 0
train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=SEED)

context_sets = get_context_sets(test_tasks)
head = InContextHead(context_sets, 10, device="cuda")
head.set_context(0)
model = models.get_mnist_model(head=head, device="cuda", seed=SEED)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

Tasks: [[7, 8], [0, 9], [1, 2], [3, 6], [4, 5]]


In [7]:
smooth_trainer = EWCSmoothTrainer(
    model,
    400,
    0.9,
    paradigm="TIL",
    seed=SEED
)

smooth_trainer.train(
    train_tasks[0],
    val_tasks[0],
    batch_size=64,
    epochs=5,
    lr=0.01,
    context_id=0
)
smooth_trainer.test(test_tasks, context_list=list(range(len(test_tasks))))

smooth_trainer.compute_lid(test_tasks[0], context_id=0)

smooth_trainer.train(
    train_tasks[1],
    val_tasks[1],
    batch_size=64,
    epochs=5,
    lr=0.01,
    context_id=1
)
smooth_trainer.test(test_tasks, context_list=list(range(len(test_tasks))))


Training Epochs: 100%|█| 5/5 [00:06<00:00,  1.31s/it, train_loss=0.0007, val_los


Test Results: [(0.0188, 0.9945), (2.2152, 0.3434), (2.382, 0.2607), (2.427, 0.4273), (2.448, 0.2556)] (Avg: (1.8982, 0.4563))
torch.Size([2002, 10])
torch.Size([2002])


100%|████████████████████████████████████████| 100/100 [00:00<00:00, 628.10it/s]


Certified L2 radius: 1.7964
1.7963845123300646


100%|████████████████████████████████████████| 100/100 [00:00<00:00, 626.76it/s]


...

100%|████████████████████████████████████████| 100/100 [00:00<00:00, 626.01it/s]


Certified L2 radius: 32.5450
32.54499052353616
Max rad:  40.076798661678026  inflate:  25
Last rad:  32.54499052353616  last inflate:  35


Training Epochs: 100%|█| 5/5 [00:11<00:00,  2.25s/it, train_loss=1.0399, val_los


Test Results: [(0.0269, 0.992), (1.002, 0.912), (2.3963, 0.2713), (2.4564, 0.4187), (2.4914, 0.2465)] (Avg: (1.6746, 0.5681))


[(0.0269, 0.992),
 (1.002, 0.912),
 (2.3963, 0.2713),
 (2.4564, 0.4187),
 (2.4914, 0.2465)]

### Domain Incremental Learning

In [3]:
SEED = 0
train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=SEED)

model = models.get_mnist_model(device="cuda", output_dim=2, seed=SEED)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

Tasks: [[7, 8], [0, 9], [1, 2], [3, 6], [4, 5]]


In [4]:
def domain_map_fn(labels: torch.Tensor) -> torch.Tensor:
    """Map the global label to the in context label."""
    return labels % 2

In [5]:
smooth_trainer = EWCSmoothTrainer(
    model,
    400,
    0.9,
    paradigm="DIL",
    seed=SEED,
    domain_map_fn=domain_map_fn,
    smooth_cheat=1_000,
    smooth_delta = 0.1
)

smooth_trainer.train(
    train_tasks[0],
    val_tasks[0],
    batch_size=64,
    epochs=5,
    lr=0.01,
)
smooth_trainer.test(test_tasks)

smooth_trainer.compute_lid(test_tasks[0])

smooth_trainer.train(
    train_tasks[1],
    val_tasks[1],
    batch_size=64,
    epochs=5,
    lr=0.01,
)
smooth_trainer.test(test_tasks)


Training Epochs:   0%|                                    | 0/5 [00:00<?, ?it/s]

Training Epochs: 100%|█| 5/5 [00:06<00:00,  1.36s/it, train_loss=0.0000, val_los


Test Results: [(0.0122, 0.9945), (0.923, 0.7743), (2.5359, 0.4075), (3.0403, 0.5447), (6.829, 0.199)] (Avg: (2.6681, 0.5840))
torch.Size([2002, 2])
torch.Size([2002])


100%|████████████████████████████████████████| 100/100 [00:00<00:00, 555.43it/s]


Certified L2 radius: 1.8880
1.8879988918577366


...

100%|████████████████████████████████████████| 100/100 [00:00<00:00, 630.18it/s]


Certified L2 radius: 63.6540
63.65404258170151
Max rad:  77.06610746531724  inflate:  50
Last rad:  63.65404258170151  last inflate:  65


Training Epochs: 100%|█| 5/5 [00:11<00:00,  2.24s/it, train_loss=0.0157, val_los


Test Results: [(0.2629, 0.9041), (0.0373, 0.9899), (1.1732, 0.6751), (0.321, 0.9019), (5.1728, 0.2316)] (Avg: (1.3934, 0.7405))


[(0.2629, 0.9041),
 (0.0373, 0.9899),
 (1.1732, 0.6751),
 (0.321, 0.9019),
 (5.1728, 0.2316)]

### Class Incremental Learning

In [3]:
SEED = 0
train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=SEED)

model = models.get_mnist_model(device="cuda", output_dim=10, seed=SEED)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

Tasks: [[7, 8], [0, 9], [1, 2], [3, 6], [4, 5]]


In [4]:
smooth_trainer = EWCSmoothTrainer(
    model,
    400,
    0.9,
    paradigm="CIL",
    seed=SEED,
    smooth_cheat=1_000,
    smooth_delta = 0.1
)

smooth_trainer.train(
    train_tasks[0],
    val_tasks[0],
    batch_size=64,
    epochs=5,
    lr=0.01,
)
smooth_trainer.test(test_tasks)

smooth_trainer.compute_lid(test_tasks[0])

smooth_trainer.train(
    train_tasks[1],
    val_tasks[1],
    batch_size=64,
    epochs=5,
    lr=0.01,
)
smooth_trainer.test(test_tasks)


Training Epochs:   0%|                                    | 0/5 [00:00<?, ?it/s]

Training Epochs: 100%|█| 5/5 [00:06<00:00,  1.38s/it, train_loss=0.0001, val_los


Test Results: [(0.0106, 0.9965), (42.3224, 0.0), (43.194, 0.0), (46.1687, 0.0), (44.7789, 0.0)] (Avg: (35.2949, 0.1993))
torch.Size([2002, 10])
torch.Size([2002])


100%|████████████████████████████████████████| 100/100 [00:00<00:00, 553.69it/s]


Certified L2 radius: 1.8880
1.8879988918577366


...

100%|████████████████████████████████████████| 100/100 [00:00<00:00, 627.32it/s]


Certified L2 radius: 46.6320
46.63202596863991
Max rad:  58.77082317773803  inflate:  35
Last rad:  46.63202596863991  last inflate:  50


Training Epochs: 100%|█| 5/5 [00:11<00:00,  2.26s/it, train_loss=6.2478, val_los


Test Results: [(0.0639, 0.988), (6.1176, 0.0), (9.9333, 0.0), (12.0917, 0.0), (10.8807, 0.0)] (Avg: (7.8174, 0.1976))


[(0.0639, 0.988), (6.1176, 0.0), (9.9333, 0.0), (12.0917, 0.0), (10.8807, 0.0)]